<a href="https://colab.research.google.com/github/pranta2003/Fetal-health-multimodal-prediction/blob/master/Capstone_fetalDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2 — NOW verify your files, using the full path as data
import os

data_path = "/content/drive/MyDrive/fetal-health-project/data/raw"

expected_files = []
for yr in range(2016, 2024):
    expected_files.append(f"{yr}_NATAL.csv")
    expected_files.append(f"{yr}_FETAL_COD.csv")

print(f"Checking {data_path}\n")
missing = []
for fname in expected_files:
    full_path = os.path.join(data_path, fname)
    if os.path.exists(full_path):
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"gotcha {fname} — {size_mb:.1f} MB")
    else:
        print(f"not find MISSING: {fname}")
        missing.append(fname)

print(f"\n{'All 16 files found!' if not missing else f'{len(missing)} files still missing.'}")

Checking /content/drive/MyDrive/fetal-health-project/data/raw

gotcha 2016_NATAL.csv — 5972.4 MB
gotcha 2016_FETAL_COD.csv — 15.6 MB
gotcha 2017_NATAL.csv — 5945.1 MB
gotcha 2017_FETAL_COD.csv — 14.9 MB
gotcha 2018_NATAL.csv — 5855.1 MB
gotcha 2018_FETAL_COD.csv — 13.7 MB
gotcha 2019_NATAL.csv — 5787.4 MB
gotcha 2019_FETAL_COD.csv — 13.4 MB
gotcha 2020_NATAL.csv — 5575.2 MB
gotcha 2020_FETAL_COD.csv — 12.2 MB
gotcha 2021_NATAL.csv — 5652.4 MB
gotcha 2021_FETAL_COD.csv — 12.5 MB
gotcha 2022_NATAL.csv — 5661.8 MB
gotcha 2022_FETAL_COD.csv — 11.9 MB
gotcha 2023_NATAL.csv — 5552.5 MB
gotcha 2023_FETAL_COD.csv — 11.7 MB

All 16 files found!


In [ ]:
import pandas as pd
import os

# Session crashed, so restart your runtime first (Runtime > Restart session),
# then re-mount Drive before running this cell

os.makedirs("/content/drive/MyDrive/fetal-health-project/data/processed", exist_ok=True)

years = range(2016, 2024)

WANTED_COLS = ['ATTEND', 'BFACIL', 'BFACIL3', 'BMI', 'BMI_R', 'BWTR4', 'CIG_0', 'CIG_1',
    'CIG_2', 'CIG_3', 'CIG_REC', 'COMBGEST', 'DBWT', 'DLMP_MM', 'DLMP_YY', 'DMETH_REC',
    'DPLURAL', 'FAGECOMB', 'FAGEREC11', 'FAGERPT_FLG', 'F_CIGS_0', 'F_CIGS_1', 'F_CIGS_2',
    'F_CIGS_3', 'F_MEDUC', 'F_MM_AICU', 'F_MPCB', 'F_M_HT', 'F_PWGT', 'F_RF_CESAR',
    'F_RF_GDIAB', 'F_RF_GHYPER', 'F_RF_NCESAR', 'F_RF_PDIAB', 'F_RF_PHYPER', 'F_TOBACO',
    'F_WIC', 'ILLB_R', 'ILLB_R11', 'IMP_PLUR', 'IMP_SEX', 'LBO_REC', 'MAGER', 'MAGER14',
    'MAGER9', 'MAGE_IMPFLG', 'MAGE_REPFLG', 'MBSTATE_REC', 'MEDUC', 'ME_PRES', 'ME_ROUT',
    'ME_TRIAL', 'MHISPX', 'MM_AICU', 'MM_RUPT', 'MRACE15', 'MRACE6', 'MRACEHISP',
    'MRACEIMP', 'M_Ht_In', 'OBGEST_FLG', 'OEGest_Comb', 'PRECARE', 'PRIORDEAD',
    'PRIORLIVE', 'PWgt_R', 'RDMETH_REC', 'RESTATUS', 'RF_ARTEC', 'RF_CESAR', 'RF_CESARN',
    'RF_EHYPE', 'RF_FEDRG', 'RF_GDIAB', 'RF_GHYPE', 'RF_INFTR', 'SEX', 'WIC']

CHUNK_SIZE = 200_000
SAMPLE_PER_YEAR = 45_000

def get_available_cols(path, wanted_cols):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    available = [c for c in wanted_cols if c in header]
    missing = [c for c in wanted_cols if c not in header]
    return available, missing

def count_rows_chunked(path, one_col, chunksize=CHUNK_SIZE):
    """Cheap pass — reads only 1 column to count total rows without heavy memory use"""
    total = 0
    for chunk in pd.read_csv(path, usecols=[one_col], chunksize=chunksize, low_memory=False):
        total += len(chunk)
    return total

def sample_csv_chunked(path, available_cols, missing_cols, target_n, chunksize=CHUNK_SIZE):
    """Reads the file in chunks, keeping only a small random slice of each chunk —
    the full file is NEVER held in memory at once, which is what fixes the crash"""
    total_rows = count_rows_chunked(path, available_cols[0], chunksize)
    frac = min(1.0, target_n / total_rows) if total_rows > 0 else 0

    sampled_chunks = []
    for chunk in pd.read_csv(path, usecols=available_cols, chunksize=chunksize, low_memory=False):
        if frac < 1.0:
            chunk = chunk.sample(frac=frac, random_state=42)
        sampled_chunks.append(chunk)

    result = pd.concat(sampled_chunks, ignore_index=True)
    if len(result) > target_n:
        result = result.sample(n=target_n, random_state=42)
    for col in missing_cols:
        result[col] = pd.NA
    return result, total_rows

birth_frames = []
for yr in years:
    path = f"/content/drive/MyDrive/fetal-health-project/data/raw/{yr}_NATAL.csv"
    available, missing = get_available_cols(path, WANTED_COLS)
    if missing:
        print(f"{yr}: missing columns filled as NaN → {missing}")
    df_sample, total_rows = sample_csv_chunked(path, available, missing, SAMPLE_PER_YEAR)
    df_sample["OUTCOME"] = 0
    df_sample["YEAR"] = yr
    birth_frames.append(df_sample)
    print(f"{yr}: sampled {len(df_sample)} rows out of {total_rows:,} total")
    del df_sample

livebirth_data = pd.concat(birth_frames, ignore_index=True)
print(f"\nTotal live birth rows: {len(livebirth_data)}")
livebirth_data.to_csv("/content/drive/MyDrive/fetal-health-project/data/processed/livebirth_sampled_all_years.csv", index=False)
print("✅ Saved successfully")

2016: missing columns filled as NaN → ['F_RF_CESAR', 'F_RF_NCESAR', 'MHISPX', 'RF_CESAR', 'RF_CESARN']
2016: sampled 45000 rows out of 3,956,112 total
2017: missing columns filled as NaN → ['MHISPX']
2017: sampled 45000 rows out of 3,864,754 total
2018: sampled 44991 rows out of 3,801,534 total
2019: sampled 44997 rows out of 3,757,582 total
2020: sampled 44994 rows out of 3,619,826 total
2021: sampled 44993 rows out of 3,669,928 total
2022: sampled 44995 rows out of 3,676,029 total
2023: sampled 44991 rows out of 3,605,081 total

Total live birth rows: 359961
✅ Saved successfully


In [ ]:
stillbirth = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/stillbirth_all_years.csv")
livebirth = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/livebirth_sampled_all_years.csv")

final = pd.concat([stillbirth, livebirth], ignore_index=True)
final = final.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final shape:", final.shape)
print(final["OUTCOME"].value_counts())
print("\nMissing values per column (top 10):")
print(final.isna().sum().sort_values(ascending=False).head(10))

final.to_csv("/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv", index=False)
print("\n Saved clinical_training_data.csv — ready for train_clinical_model.py")

/tmp/ipykernel_738/1881883685.py:2: DtypeWarning: Columns (76) have mixed types. Specify dtype option on import or set low_memory=False.
  livebirth = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/livebirth_sampled_all_years.csv")


Final shape: (718136, 80)
OUTCOME
0    359961
1    358175
Name: count, dtype: int64

Missing values per column (top 10):
MAGE_IMPFLG    356422
FAGERPT_FLG    356102
IMP_PLUR       355693
MAGE_REPFLG    352380
IMP_SEX        352316
MRACEIMP       302416
OBGEST_FLG     290891
MHISPX          90000
RF_CESAR        52691
RF_CESARN       52691
dtype: int64

 Saved clinical_training_data.csv — ready for train_clinical_model.py


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv")
print("Shape:", df.shape)
print("Class balance:\n", df["OUTCOME"].value_counts(normalize=True))
print("\nSample rows:\n", df.sample(5))

Shape: (718136, 80)
Class balance:
 OUTCOME
0    0.501243
1    0.498757
Name: proportion, dtype: float64

Sample rows:
         BFACIL  BFACIL3 MAGE_IMPFLG MAGE_REPFLG  MAGER  MAGER14  MAGER9  \
694147     1.0        1         NaN         NaN     31       10       5   
217633     1.0        1         NaN         NaN     43       12       7   
246335     1.0        1                             30       10       5   
409123     1.0        1                             33       10       5   
576641     1.0        1                             25        9       4   

        MBSTATE_REC  RESTATUS  MRACE6  ...  F_RF_CESAR F_RF_NCESAR  F_MM_AICU  \
694147            1         2     2.0  ...         1.0         1.0        1.0   
217633            1         2     1.0  ...         1.0         1.0        1.0   
246335            1         2     1.0  ...         1.0         1.0        1.0   
409123            1         1     2.0  ...         1.0         1.0        1.0   
576641            1     

In [ ]:
# Paste this whole script into a new Colab cell — it already has the
# correct Drive path built in, ready to run as-is

DATA_PATH = "/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv"
MODEL_OUT_PATH = "/content/drive/MyDrive/fetal-health-project/models/stream3_clinical_xgb.json"

# --- paste the rest of the script's contents below this line ---

In [9]:
#Stram - 3 value check and AUC check from tabular dataset

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import os

RANDOM_SEED = 42
DATA_PATH = "/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv"
MODEL_OUT_PATH = "/content/drive/MyDrive/fetal-health-project/models/stream3_clinical_xgb.json"

TARGET_COLUMN = "OUTCOME"

# Retrain WITHOUT COMBGEST to get an honest "floor" performance number
# using only genuine maternal risk factors, no registry-timing artifact

NUMERIC_FEATURES_NO_GA = ["MAGER", "BMI", "PRECARE", "PRIORDEAD", "PRIORLIVE", "MEDUC"]

X_no_ga = df[NUMERIC_FEATURES_NO_GA + CATEGORICAL_FEATURES]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X_no_ga, y, test_size=0.2, random_state=42, stratify=y)

# reuse your existing build_pipeline(), just swap which NUMERIC_FEATURES list it references



CATEGORICAL_FEATURES = [
    "RF_GDIAB", "RF_GHYPE", "RF_EHYPE", "RF_CESAR", "RF_ARTEC", "RF_INFTR",
    "RF_FEDRG", "CIG_REC", "WIC", "DPLURAL", "SEX", "MRACE6",
    "MBSTATE_REC",
]

UNKNOWN_CODES = ["U", "9", "99", "Unknown"]
NUMERIC_UNKNOWN_CODES = {"COMBGEST": [99]}

def load_data(path=DATA_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {path}.")
    df = pd.read_csv(path)
    print(f"Loaded combined clinical dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    missing_needed = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COLUMN]
                       if c not in df.columns]
    if missing_needed:
        raise KeyError(f"Expected column(s) not found: {missing_needed}")
    return df

def clean_unknown_codes(df):
    before_counts = {col: (df[col].isin(UNKNOWN_CODES)).sum() for col in CATEGORICAL_FEATURES}
    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].replace(UNKNOWN_CODES, np.nan)
    print(f"Replaced {sum(before_counts.values())} Unknown-coded categorical values")
    for col, codes in NUMERIC_UNKNOWN_CODES.items():
        before = df[col].isin(codes).sum()
        df[col] = df[col].replace(codes, np.nan)
        print(f"{col}: {before} sentinel values {codes} converted to NaN")
    return df

def build_pipeline():
    preprocessor = ColumnTransformer(transformers=[
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("categorical", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL_FEATURES),
    ])
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=RANDOM_SEED,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])

def train(df):
    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y = df[TARGET_COLUMN]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)[:, 1]
    print("\n--- Stream 3 Clinical Risk Model: Evaluation ---")
    print(classification_report(y_test, preds, target_names=["Healthy/Live Birth", "Stillbirth"]))
    print(f"AUC-ROC: {roc_auc_score(y_test, proba):.4f}")
    return pipeline

df = load_data()
df = clean_unknown_codes(df)

print("\nCOMBGEST by outcome, after cleaning:")
print(df.groupby("OUTCOME")["COMBGEST"].describe())

pipeline = train(df)

os.makedirs(os.path.dirname(MODEL_OUT_PATH), exist_ok=True)
pipeline.named_steps["model"].save_model(MODEL_OUT_PATH)
print(f"\nModel saved to {MODEL_OUT_PATH}")

Loaded combined clinical dataset: 718136 rows, 80 columns
Replaced 1039563 Unknown-coded categorical values
COMBGEST: 12188 sentinel values [99] converted to NaN

COMBGEST by outcome, after cleaning:
            count       mean        std   min   25%   50%   75%   max
OUTCOME                                                              
0        359685.0  38.559067   2.475167  17.0  38.0  39.0  40.0  47.0
1        346263.0  19.604070  10.764422   2.0  10.0  20.0  28.0  47.0

--- Stream 3 Clinical Risk Model: Evaluation ---
                    precision    recall  f1-score   support

Healthy/Live Birth       0.90      0.95      0.92     71993
        Stillbirth       0.95      0.89      0.92     71635

          accuracy                           0.92    143628
         macro avg       0.92      0.92      0.92    143628
      weighted avg       0.92      0.92      0.92    143628

AUC-ROC: 0.9684

Model saved to /content/drive/MyDrive/fetal-health-project/models/stream3_clinical_xgb.jso

In [2]:
feature_names = (NUMERIC_FEATURES +
    list(pipeline.named_steps["preprocess"].transformers_[1][1]
         .named_steps["onehot"].get_feature_names_out(CATEGORICAL_FEATURES)))

importances = pipeline.named_steps["model"].feature_importances_
top_15 = sorted(zip(feature_names, importances), key=lambda x: -x[1])[:15]

for name, score in top_15:
    print(f"{name}: {score:.4f}")

COMBGEST: 0.4336
DMETH_REC_9: 0.1361
PRIORDEAD: 0.0890
PRECARE: 0.0872
MBSTATE_REC_3: 0.0393
DMETH_REC_2: 0.0367
MEDUC: 0.0287
WIC_N: 0.0155
DMETH_REC_1: 0.0119
WIC_Y: 0.0112
DPLURAL_1: 0.0109
CIG_REC_Y: 0.0079
MRACE6_2.0: 0.0075
BMI: 0.0068
CIG_REC_N: 0.0062


In [6]:
print(df[df["OUTCOME"] == 1]["COMBGEST"].value_counts().sort_index().head(20))

COMBGEST
2.0       962
3.0      1442
4.0      3410
5.0      7115
6.0     14921
7.0     13973
8.0     21057
9.0     20148
10.0    19002
11.0    14487
12.0    11676
13.0     7840
14.0     5939
15.0     5475
16.0     6216
17.0     6211
18.0     6298
19.0     6926
20.0    16564
21.0    15884
Name: count, dtype: int64


In [8]:
#Layer - 3 , Hadlock Formula applying


def calculate_efw(hc_mm, bpd_mm, ac_mm, fl_mm):
    """Hadlock 1985 four-parameter EFW formula.
    Inputs in millimeters (matching your pipeline's standardized units) —
    internally converted to cm, since Hadlock's coefficients are cm-calibrated."""
    hc_cm, bpd_cm, ac_cm, fl_cm = hc_mm/10, bpd_mm/10, ac_mm/10, fl_mm/10
    log10_efw = (
        1.3596
        - 0.00386 * (ac_cm * fl_cm)
        + 0.0064 * hc_cm
        + 0.00061 * (bpd_cm * ac_cm)
        + 0.0424 * ac_cm
        + 0.174 * fl_cm
    )
    return 10 ** log10_efw

efw = calculate_efw(hc_mm=280, bpd_mm=80, ac_mm=250, fl_mm=55)
print(f"Test EFW: {efw:.1f} grams")  # should print ~1403.0

Test EFW: 1403.0 grams


In [11]:
import math
from scipy.stats import norm

def calculate_efw(hc_mm, bpd_mm, ac_mm, fl_mm):
    """Hadlock 1985 four-parameter EFW formula. Inputs in mm, converted
    to cm internally since Hadlock's coefficients are cm-calibrated."""
    hc_cm, bpd_cm, ac_cm, fl_cm = hc_mm/10, bpd_mm/10, ac_mm/10, fl_mm/10
    log10_efw = (1.3596 - 0.00386*(ac_cm*fl_cm) + 0.0064*hc_cm
                 + 0.00061*(bpd_cm*ac_cm) + 0.0424*ac_cm + 0.174*fl_cm)
    return 10 ** log10_efw

def hadlock_reference_mean_sd(ga_weeks):
    """Hadlock et al. 1991 fetal weight reference standard.
    Mean EFW from the paper's published regression equation;
    SD as a uniform 12.7% coefficient of variation, per the paper."""
    mean = math.exp(0.578 + 0.332*ga_weeks - 0.00354*(ga_weeks**2))
    sd = 0.127 * mean
    return mean, sd

def growth_percentile(efw_grams, ga_weeks):
    """Converts an EFW at a given gestational age into a percentile
    against the Hadlock 1991 reference standard."""
    mean, sd = hadlock_reference_mean_sd(ga_weeks)
    z = (efw_grams - mean) / sd
    percentile = norm.cdf(z) * 100
    return percentile, z

def flag_growth_restriction(percentile, threshold=10):
    """Standard clinical convention: <10th percentile = growth-restricted."""
    return percentile < threshold